In [1]:
import os, sys, time, importlib.util
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [ ]:
_pyc_dir = os.path.abspath('__pycache__')def _load_pyc(name):    pyc = os.path.join(_pyc_dir, f'{name}.cpython-310.pyc')    spec = importlib.util.spec_from_file_location(name, pyc)    mod = importlib.util.module_from_spec(spec)    spec.loader.exec_module(mod)    sys.modules[name] = mod_load_pyc('tuco_dataset')from tuco_dataset import TucoDatasetfrom torch.utils.data import SubsetDATA_ROOT = '../Data/Tuco'MAX_VAL = Noneval_ds = TucoDataset(DATA_ROOT, split='val')if MAX_VAL is not None:    val_ds = Subset(val_ds, range(min(MAX_VAL, len(val_ds))))val_moving, val_fixed = [], []val_moving_seg, val_fixed_seg = [], []for s in val_ds:    val_moving.append(s['moving'])    val_fixed.append(s['fixed'])    val_moving_seg.append(s['moving_seg'])    val_fixed_seg.append(s['fixed_seg'])val_moving = torch.stack(val_moving)val_fixed = torch.stack(val_fixed)val_moving_seg = torch.stack(val_moving_seg)val_fixed_seg = torch.stack(val_fixed_seg)N, _, D, H, W = val_moving.shapeprint(f"Loaded {N} val subjects  |  volume shape: D={D} H={H} W={W}")# Per MambaMorph: ignore labels 0 (background), 5 and 24 (noisy regions)SEG_LABELS = [2, 3, 4, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 26, 28]N_LABELS = len(SEG_LABELS)print(f"{N_LABELS} anatomical labels (ignoring 0, 5, 24)")np.random.seed(42)indices = np.random.permutation(N)n_pairs = N // 2pairings = [(indices[2*i], indices[2*i+1]) for i in range(n_pairs)]print(f"{n_pairs} cross-subject pairs: {pairings}")

In [ ]:
class SpatialTransformer2D(nn.Module):    def forward(self, src, flow):        b, _, h, w = src.shape        yy, xx = torch.meshgrid(torch.arange(h, device=src.device),                                torch.arange(w, device=src.device), indexing='ij')        bg = torch.stack((xx, yy), 0).float().unsqueeze(0).repeat(b, 1, 1, 1).to(flow)        pts = bg + flow        gx = 2*(pts[:,0]/(w-1))-1;  gy = 2*(pts[:,1]/(h-1))-1        return F.grid_sample(src, torch.stack((gx, gy), -1),                             mode='bilinear', padding_mode='border', align_corners=True)class SpatialTransformer3D(nn.Module):    def forward(self, src, flow):        b, _, d, h, w = src.shape        zz, yy, xx = torch.meshgrid(torch.arange(d, device=src.device),                                    torch.arange(h, device=src.device),                                    torch.arange(w, device=src.device), indexing='ij')        bg = torch.stack((xx, yy, zz), 0).float().unsqueeze(0).repeat(b, 1, 1, 1, 1).to(flow)        pts = bg + flow        gx=2*(pts[:,0]/(w-1))-1; gy=2*(pts[:,1]/(h-1))-1; gz=2*(pts[:,2]/(d-1))-1        return F.grid_sample(src, torch.stack((gx, gy, gz), -1),                             mode='bilinear', padding_mode='border', align_corners=True)class ConvBlock2D(nn.Module):    def __init__(self, in_channels, out_channels):        super().__init__()        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)        self.act = nn.LeakyReLU(0.2)    def forward(self, x):        return self.act(self.conv(x))class ConvBlock3D(nn.Module):    def __init__(self, in_channels, out_channels):        super().__init__()        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1)        self.act = nn.LeakyReLU(0.2)    def forward(self, x):        return self.act(self.conv(x))class VxmDense3D(nn.Module):    def __init__(self, enc_feats=(16, 32, 32, 32), final_feats=(32, 16)):        super().__init__()        self.encoders = nn.ModuleList()        in_ch = 2        for nf in enc_feats:            self.encoders.append(ConvBlock3D(in_ch, nf)); in_ch = nf        self.pool = nn.MaxPool3d(2)        self.bottleneck = ConvBlock3D(enc_feats[-1], enc_feats[-1])        self.decoders = nn.ModuleList()        dec_in = enc_feats[-1]        for sk in reversed(enc_feats):            self.decoders.append(ConvBlock3D(dec_in + sk, sk)); dec_in = sk        self.final_conv0 = ConvBlock3D(dec_in, final_feats[0])        self.final_conv1 = ConvBlock3D(final_feats[0], final_feats[1])        self.flow = nn.Conv3d(final_feats[1], 3, kernel_size=3, padding=1)        self.transformer = SpatialTransformer3D()        self.apply(self._iw)        nn.init.zeros_(self.flow.weight); nn.init.zeros_(self.flow.bias)    @staticmethod    def _iw(m):        if isinstance(m, nn.Conv3d):            nn.init.kaiming_normal_(m.weight, nonlinearity='leaky_relu')            if m.bias is not None: nn.init.zeros_(m.bias)    def forward(self, src, tgt):        x = torch.cat([src, tgt], dim=1); skips = []        for enc in self.encoders:            x = enc(x); skips.append(x); x = self.pool(x)        x = self.bottleneck(x)        for skip, dec in zip(reversed(skips), self.decoders):            x = F.interpolate(x, scale_factor=2, mode='trilinear', align_corners=True)            x = torch.cat([x, skip], dim=1); x = dec(x)        x = self.final_conv0(x); x = self.final_conv1(x)        flow = self.flow(x)        return self.transformer(src, flow), flowclass VxmDense2p5D(nn.Module):    def __init__(self, n_stack, enc_feats=(16, 32, 32, 32), final_feats=(32, 16)):        super().__init__()        self.n_stack = n_stack        self.center = n_stack // 2        self.encoders = nn.ModuleList()        in_ch = 2 * n_stack        for nf in enc_feats:            self.encoders.append(ConvBlock2D(in_ch, nf)); in_ch = nf        self.pool = nn.MaxPool2d(2)        self.bottleneck = ConvBlock2D(enc_feats[-1], enc_feats[-1])        self.decoders = nn.ModuleList()        dec_in = enc_feats[-1]        for sk in reversed(enc_feats):            self.decoders.append(ConvBlock2D(dec_in + sk, sk)); dec_in = sk        self.final_conv0 = ConvBlock2D(dec_in, final_feats[0])        self.final_conv1 = ConvBlock2D(final_feats[0], final_feats[1])        self.flow = nn.Conv2d(final_feats[1], 2, kernel_size=3, padding=1)        self.transformer = SpatialTransformer2D()        self.apply(self._iw)        nn.init.zeros_(self.flow.weight); nn.init.zeros_(self.flow.bias)    @staticmethod    def _iw(m):        if isinstance(m, nn.Conv2d):            nn.init.kaiming_normal_(m.weight, nonlinearity='leaky_relu')            if m.bias is not None: nn.init.zeros_(m.bias)    def forward(self, src_stack, tgt_stack):        x = torch.cat([src_stack, tgt_stack], dim=1); skips = []        for enc in self.encoders:            x = enc(x); skips.append(x); x = self.pool(x)        x = self.bottleneck(x)        for skip, dec in zip(reversed(skips), self.decoders):            x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)            x = torch.cat([x, skip], dim=1); x = dec(x)        x = self.final_conv0(x); x = self.final_conv1(x)        flow = self.flow(x)        src_center = src_stack[:, self.center : self.center + 1]        return self.transformer(src_center, flow), flow

In [ ]:
WEIGHT_DIR = 'trained_weights'N_STACK = 7WR = N_STACK // 2model_3d = VxmDense3D().to(device)model_2p5d = VxmDense2p5D(n_stack=N_STACK).to(device)def _load(model, path):    full = os.path.join(WEIGHT_DIR, path)    if not os.path.exists(full):        print(f"  [WARN] {full} not found")        return False    model.load_state_dict(torch.load(full, map_location=device))    model.eval()    size_mb = os.path.getsize(full) / 1e6    n_params = sum(p.numel() for p in model.parameters()) / 1e6    print(f"  Loaded {full:50s}  {n_params:.3f}M params  {size_mb:.2f} MB")    return Trueprint("Loading weights:")ok_3d = _load(model_3d,   '3d_dense_best.pth')ok_2p5d = _load(model_2p5d, '2p5d_dense_best.pth')

In [ ]:
DOWNSAMPLE = Trueif DOWNSAMPLE:    ORIENT_CONFIG = [        (0, 96,  'axial',    (112, 96)),        (1, 112, 'coronal',  (96,  96)),        (2, 96,  'sagittal', (96,  112)),    ]else:    ORIENT_CONFIG = [        (0, 176, 'axial',    (208, 192)),        (1, 208, 'coronal',  (176, 192)),        (2, 192, 'sagittal', (176, 208)),    ]def preprocess_3d(vol):    x = vol.to(device)    x = F.interpolate(x, scale_factor=0.5, mode='trilinear', align_corners=False)    x = F.pad(x, (0,0, 4,4, 4,4))    return xdef preprocess_seg_3d(seg):    x = seg.float().to(device)    x = F.interpolate(x, scale_factor=0.5, mode='nearest')    x = F.pad(x, (0,0, 4,4, 4,4))    return xdef seg_to_onehot(seg, labels=SEG_LABELS):    seg_squeezed = seg[:, 0].long()    onehot = torch.zeros(seg.shape[0], len(labels), *seg.shape[2:],                         device=seg.device, dtype=torch.float32)    for i, lbl in enumerate(labels):        onehot[:, i] = (seg_squeezed == lbl).float()    return onehotdef seg_to_onehot_2d(seg, labels=SEG_LABELS):    seg_long = seg.long()    onehot = torch.zeros(seg.shape[0], len(labels), *seg.shape[1:],                         device=seg.device, dtype=torch.float32)    for i, lbl in enumerate(labels):        onehot[:, i] = (seg_long == lbl).float()    return onehotclass NearestSpatialTransformer3D(nn.Module):    def forward(self, src, flow):        b, _, d, h, w = src.shape        device = src.device        zz, yy, xx = torch.meshgrid(            torch.arange(d, device=device),            torch.arange(h, device=device),            torch.arange(w, device=device), indexing='ij')        base_grid = torch.stack((xx, yy, zz), dim=0).float().unsqueeze(0).expand(b, -1, -1, -1, -1)        pts = base_grid + flow        x = 2.0 * (pts[:, 0] / (w - 1.0)) - 1.0        y = 2.0 * (pts[:, 1] / (h - 1.0)) - 1.0        z = 2.0 * (pts[:, 2] / (d - 1.0)) - 1.0        grid = torch.stack((x, y, z), dim=-1)        return F.grid_sample(src, grid, mode='nearest', padding_mode='border', align_corners=True)nearest_stn_3d = NearestSpatialTransformer3D().to(device)@torch.no_grad()def infer_3d(moving_vol, fixed_vol, moving_seg, fixed_seg):    D_orig, H_orig, W_orig = moving_vol.shape[2:]    mv_ds = preprocess_3d(moving_vol)    fx_ds = preprocess_3d(fixed_vol)    ms_ds = preprocess_seg_3d(moving_seg)    fs_ds = preprocess_seg_3d(fixed_seg)    torch.cuda.synchronize()    torch.cuda.reset_peak_memory_stats()    t0 = time.perf_counter()    moved_ds, flow_ds = model_3d(mv_ds, fx_ds)    torch.cuda.synchronize()    wall_s = time.perf_counter() - t0    peak_mb = torch.cuda.max_memory_allocated() / 1e6    ms_oh = seg_to_onehot(ms_ds)    warped_seg_oh = nearest_stn_3d(ms_oh, flow_ds)    warped_seg_np = warped_seg_oh[0].cpu().numpy()    warped_labels = np.zeros(warped_seg_np.shape[1:], dtype=np.int16)    for li, lbl in enumerate(SEG_LABELS):        warped_labels[warped_seg_np[li] > 0.5] = lbl    fixed_seg_ds_np = fs_ds[0, 0].cpu().numpy().astype(np.int16)    moved_cropped = moved_ds[:, :, 4:-4, 4:-4, :]    moved_full = F.interpolate(moved_cropped, size=(D_orig, H_orig, W_orig),                               mode='trilinear', align_corners=False).cpu()    return moved_full, flow_ds.cpu(), warped_labels, fixed_seg_ds_np, wall_s, peak_mbdef _extract_stack(vol_np, axis, z, wr):    if axis == 0:        return vol_np[z-wr : z+wr+1]    elif axis == 1:        return vol_np[:, z-wr : z+wr+1, :].transpose(1, 0, 2)    else:        return vol_np[:, :, z-wr : z+wr+1].transpose(2, 0, 1)def _extract_slice(vol_np, axis, z):    if axis == 0:        return vol_np[z]    elif axis == 1:        return vol_np[:, z, :]    else:        return vol_np[:, :, z]@torch.no_grad()def infer_2p5d(moving_vol, fixed_vol, moving_seg, fixed_seg,               axis=0, batch_size=16):    if DOWNSAMPLE:        mv_t = preprocess_3d(moving_vol)        fx_t = preprocess_3d(fixed_vol)        ms_t = preprocess_seg_3d(moving_seg)        fs_t = preprocess_seg_3d(fixed_seg)        mr = mv_t[0, 0].cpu().numpy()        ct = fx_t[0, 0].cpu().numpy()        mr_seg_np = ms_t[0, 0].cpu().numpy().astype(np.int16)        fs_seg_np = fs_t[0, 0].cpu().numpy().astype(np.int16)    else:        mr = moving_vol[0, 0].numpy()        ct = fixed_vol[0, 0].numpy()        mr_seg_np = moving_seg[0, 0].numpy().astype(np.int16)        fs_seg_np = fixed_seg[0, 0].numpy().astype(np.int16)    ax_idx, D_ax, orient_name, slice_shape = ORIENT_CONFIG[axis]    wr = WR    slice_indices = list(range(wr, D_ax - wr))    flow_slices = []    torch.cuda.synchronize()    torch.cuda.reset_peak_memory_stats()    t0 = time.perf_counter()    for start in range(0, len(slice_indices), batch_size):        end = min(start + batch_size, len(slice_indices))        mr_b = np.stack([np.ascontiguousarray(_extract_stack(mr, ax_idx, slice_indices[k], wr))                         for k in range(start, end)])        ct_b = np.stack([np.ascontiguousarray(_extract_stack(ct, ax_idx, slice_indices[k], wr))                         for k in range(start, end)])        mr_t = torch.from_numpy(mr_b).to(device)        ct_t = torch.from_numpy(ct_b).to(device)        _, flow = model_2p5d(mr_t, ct_t)        flow_slices.append(flow.cpu().numpy())    torch.cuda.synchronize()    wall_s = time.perf_counter() - t0    peak_mb = torch.cuda.max_memory_allocated() / 1e6    flow_all = np.concatenate(flow_slices, 0)    warped_labels = np.zeros_like(mr_seg_np)    for si, z in enumerate(slice_indices):        flow_2d = flow_all[si]        flow_t = torch.from_numpy(flow_2d)[None].float().to(device)        seg_slice = _extract_slice(mr_seg_np, ax_idx, z)        seg_oh = seg_to_onehot_2d(            torch.from_numpy(seg_slice.astype(np.int64))[None].to(device))        _, _, h, w = seg_oh.shape        yy, xx = torch.meshgrid(            torch.arange(h, device=device), torch.arange(w, device=device), indexing='ij')        base = torch.stack((xx, yy), dim=0).float().unsqueeze(0)        pts = base + flow_t        gx = 2.0 * (pts[:, 0] / (w - 1.0)) - 1.0        gy = 2.0 * (pts[:, 1] / (h - 1.0)) - 1.0        grid = torch.stack((gx, gy), dim=-1)        warped_oh = F.grid_sample(seg_oh, grid, mode='nearest',                                  padding_mode='border', align_corners=True)        warped_np = warped_oh[0].cpu().numpy()        result_slice = np.zeros(seg_slice.shape, dtype=np.int16)        for li, lbl in enumerate(SEG_LABELS):            result_slice[warped_np[li] > 0.5] = lbl        if ax_idx == 0:            warped_labels[z] = result_slice        elif ax_idx == 1:            warped_labels[:, z, :] = result_slice        else:            warped_labels[:, :, z] = result_slice    return warped_labels, fs_seg_np, wall_s, peak_mb

In [ ]:
from scipy.ndimage import distance_transform_edt


def compute_dice_per_label(warped_seg, fixed_seg, labels=SEG_LABELS):
    dice_scores = []
    for lbl in labels:
        w = (warped_seg == lbl).astype(np.float32)
        f = (fixed_seg == lbl).astype(np.float32)
        intersection = (w * f).sum()
        union = w.sum() + f.sum()
        if union == 0:
            dice_scores.append(1.0)
        else:
            dice_scores.append(2.0 * intersection / union)
    return np.array(dice_scores)


def hausdorff_distance_95(seg1, seg2):
    if seg1.sum() == 0 and seg2.sum() == 0:
        return 0.0
    if seg1.sum() == 0 or seg2.sum() == 0:
        return np.inf
    dt2 = distance_transform_edt(~seg2.astype(bool))
    d12 = dt2[seg1.astype(bool)]
    dt1 = distance_transform_edt(~seg1.astype(bool))
    d21 = dt1[seg2.astype(bool)]
    return max(np.percentile(d12, 95), np.percentile(d21, 95))


def negative_jacobian_ratio_3d(flow_np):
    D, H, W, _ = flow_np.shape
    grid = np.mgrid[:D, :H, :W].astype(np.float32).transpose(1, 2, 3, 0)
    deform = flow_np + grid
    J = np.zeros((D, H, W, 3, 3), dtype=np.float32)
    for i in range(3):
        J[..., i, 0] = np.gradient(deform[..., i], axis=0)
        J[..., i, 1] = np.gradient(deform[..., i], axis=1)
        J[..., i, 2] = np.gradient(deform[..., i], axis=2)
    det = (J[...,0,0] * (J[...,1,1]*J[...,2,2] - J[...,1,2]*J[...,2,1]) -
           J[...,0,1] * (J[...,1,0]*J[...,2,2] - J[...,1,2]*J[...,2,0]) +
           J[...,0,2] * (J[...,1,0]*J[...,2,1] - J[...,1,1]*J[...,2,0]))
    return np.sum(det <= 0) / det.size


def negative_jacobian_ratio_2d(flow_np):
    H, W, _ = flow_np.shape
    grid_h, grid_w = np.mgrid[:H, :W].astype(np.float32)
    deform = flow_np.copy()
    deform[..., 0] += grid_h
    deform[..., 1] += grid_w
    d00 = np.gradient(deform[..., 0], axis=0)
    d01 = np.gradient(deform[..., 0], axis=1)
    d10 = np.gradient(deform[..., 1], axis=0)
    d11 = np.gradient(deform[..., 1], axis=1)
    det = d00 * d11 - d01 * d10
    return np.sum(det <= 0) / det.size


def global_ncc(a, b, eps=1e-6):
    a = a.float().flatten(); b = b.float().flatten()
    a = a - a.mean();        b = b - b.mean()
    denom = torch.sqrt((a**2).sum() * (b**2).sum()).clamp(min=eps)
    return (a * b).sum() / denom

def global_mse(a, b):
    return F.mse_loss(a.float(), b.float()).item()

In [ ]:
_dummy = torch.randn(1,1,96,112,96, device=device)with torch.no_grad(): model_3d(_dummy, _dummy)del _dummy; torch.cuda.empty_cache()orient_names = ['axial', 'coronal', 'sagittal']rows_3d = []rows_25 = {name: [] for name in orient_names}for pair_idx, (i, j) in enumerate(pairings):    print(f"\nPair {pair_idx+1}/{n_pairs}: subj {i} (MR) -> subj {j} (CT)")    mv = val_moving[i:i+1]    fx = val_fixed[j:j+1]    ms = val_moving_seg[i:i+1]    fs = val_fixed_seg[j:j+1]    moved_3d, flow_3d, wseg_3d, fseg_3d, t_3d, mem_3d = infer_3d(mv, fx, ms, fs)    dice_3d = compute_dice_per_label(wseg_3d, fseg_3d)    hd95_per_label_3d = []    for li, lbl in enumerate(SEG_LABELS):        hd = hausdorff_distance_95(            (wseg_3d == lbl).astype(np.uint8),            (fseg_3d == lbl).astype(np.uint8))        hd95_per_label_3d.append(hd)    flow_3d_np = flow_3d[0].permute(1,2,3,0).numpy()    jac_3d = negative_jacobian_ratio_3d(flow_3d_np)    rows_3d.append({        'pair': pair_idx, 'subj_mr': i, 'subj_ct': j,        'dice_per_label': dice_3d, 'dice_mean': dice_3d.mean() * 100,        'hd95': np.mean(hd95_per_label_3d), 'neg_jac': jac_3d,        'time_s': t_3d if pair_idx > 0 else None,        'peak_mb': mem_3d,    })    print(f"  3D       Dice={dice_3d.mean()*100:.2f}%  HD95={np.mean(hd95_per_label_3d):.2f}  "          f"|J|≤0={jac_3d:.6f}  t={t_3d:.2f}s")    for ax_i, orient in enumerate(orient_names):        wseg_25, fseg_25, t_25, mem_25 = infer_2p5d(mv, fx, ms, fs, axis=ax_i)        dice_25 = compute_dice_per_label(wseg_25, fseg_25)        hd95_per_label_25 = []        for li, lbl in enumerate(SEG_LABELS):            hd = hausdorff_distance_95(                (wseg_25 == lbl).astype(np.uint8),                (fseg_25 == lbl).astype(np.uint8))            hd95_per_label_25.append(hd)        rows_25[orient].append({            'pair': pair_idx, 'subj_mr': i, 'subj_ct': j,            'dice_per_label': dice_25, 'dice_mean': dice_25.mean() * 100,            'hd95': np.mean(hd95_per_label_25),            'time_s': t_25 if pair_idx > 0 else None,            'peak_mb': mem_25,        })        print(f"  2.5D-{orient:8s} Dice={dice_25.mean()*100:.2f}%  HD95={np.mean(hd95_per_label_25):.2f}  "              f"t={t_25:.2f}s")    torch.cuda.empty_cache()print(f"\nDone: {n_pairs} pairs × 4 models (3D + 3 orientations).")

In [ ]:
def _size_info(model, path):    n = sum(p.numel() for p in model.parameters())    mb = os.path.getsize(os.path.join(WEIGHT_DIR, path)) / 1e6    return n, mbn3, mb3 = _size_info(model_3d,   '3d_dense_best.pth')n25, mb25 = _size_info(model_2p5d, '2p5d_dense_best.pth')def _agg(rows, has_jac=True):    dice_means = [r['dice_mean'] for r in rows]    hd95s = [r['hd95'] for r in rows]    times = [r['time_s'] for r in rows if r['time_s'] is not None]    peaks = [r['peak_mb'] for r in rows]    result = {        'Dice_%': f"{np.mean(dice_means):.2f} +/- {np.std(dice_means):.2f}",        'HD95': f"{np.mean(hd95s):.2f} +/- {np.std(hd95s):.2f}",        'Time_s': f"{np.mean(times):.4f}" if times else "N/A",        'PeakGPU_MB': f"{np.mean(peaks):.1f}",        'dice_mean_val': np.mean(dice_means),        'dice_std_val': np.std(dice_means),        'hd95_mean_val': np.mean(hd95s),        'hd95_std_val': np.std(hd95s),        'time_mean_val': np.mean(times) if times else 0,        'peak_mean_val': np.mean(peaks),    }    if has_jac:        jacs = [r['neg_jac'] for r in rows]        result['|J|<=0'] = f"{np.mean(jacs):.6f} +/- {np.std(jacs):.6f}"        result['jac_mean_val'] = np.mean(jacs)    else:        result['|J|<=0'] = "N/A (2D)"        result['jac_mean_val'] = None    return resultagg_3d = _agg(rows_3d, has_jac=True)agg_25 = {orient: _agg(rows_25[orient], has_jac=False) for orient in orient_names}dice_3d_all = np.stack([r['dice_per_label'] for r in rows_3d])dice_25_all = {orient: np.stack([r['dice_per_label'] for r in rows_25[orient]])               for orient in orient_names}print(f"{'Metric':20s} {'3D':>16s}  {'2.5D-axial':>16s}  {'2.5D-coronal':>16s}  {'2.5D-sagittal':>16s}")print("-" * 95)for label, key in [('Avg Dice (%)', 'Dice_%'), ('HD95', 'HD95'), ('|J| <= 0', '|J|<=0'),                   ('Inference (s)', 'Time_s'), ('Peak GPU (MB)', 'PeakGPU_MB')]:    vals = [agg_3d[key]] + [agg_25[o][key] for o in orient_names]    print(f"{label:20s} {vals[0]:>16s}  {vals[1]:>16s}  {vals[2]:>16s}  {vals[3]:>16s}")print(f"{'Params (M)':20s} {n3/1e6:>16.3f}  {n25/1e6:>16.3f}  {n25/1e6:>16.3f}  {n25/1e6:>16.3f}")print(f"{'Weights (MB)':20s} {mb3:>16.2f}  {mb25:>16.2f}  {mb25:>16.2f}  {mb25:>16.2f}")print(f"\nPer-label Dice (mean% across {n_pairs} pairs):")print(f"{'Label':>7s}  {'3D':>10s}  {'axial':>10s}  {'coronal':>10s}  {'sagittal':>10s}")for li, lbl in enumerate(SEG_LABELS):    d3 = dice_3d_all[:, li].mean()*100    da = dice_25_all['axial'][:, li].mean()*100    dc = dice_25_all['coronal'][:, li].mean()*100    ds = dice_25_all['sagittal'][:, li].mean()*100    print(f"  {lbl:5d}  {d3:10.2f}  {da:10.2f}  {dc:10.2f}  {ds:10.2f}")

In [ ]:
models_all = ['3D', '2.5D-ax', '2.5D-cor', '2.5D-sag']colours_all = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']agg_list = [agg_3d, agg_25['axial'], agg_25['coronal'], agg_25['sagittal']]fig, axes = plt.subplots(2, 4, figsize=(22, 9))fig.suptitle("3D vs 2.5D VoxelMorph — Multi-Orientation Evaluation", fontsize=14, fontweight='bold')ax = axes[0, 0]vals = [a['dice_mean_val'] for a in agg_list]stds = [a['dice_std_val'] for a in agg_list]bars = ax.bar(models_all, vals, yerr=stds, capsize=4, color=colours_all, width=0.6)ax.set_title('Avg Dice % (higher better)'); ax.set_ylabel('Dice %')best = int(np.argmax(vals))bars[best].set_edgecolor('black'); bars[best].set_linewidth(2)for bar, val in zip(bars, vals):    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01, f'{val:.1f}', ha='center', va='bottom', fontsize=8)ax = axes[0, 1]vals = [a['hd95_mean_val'] for a in agg_list]stds = [a['hd95_std_val'] for a in agg_list]bars = ax.bar(models_all, vals, yerr=stds, capsize=4, color=colours_all, width=0.6)ax.set_title('HD95 (lower better)'); ax.set_ylabel('HD95')best = int(np.argmin(vals))bars[best].set_edgecolor('black'); bars[best].set_linewidth(2)for bar, val in zip(bars, vals):    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01, f'{val:.2f}', ha='center', va='bottom', fontsize=8)ax = axes[0, 2]vals = [a['time_mean_val'] for a in agg_list]bars = ax.bar(models_all, vals, color=colours_all, width=0.6)ax.set_title('Inference time (s, lower better)'); ax.set_ylabel('Seconds')best = int(np.argmin(vals))bars[best].set_edgecolor('black'); bars[best].set_linewidth(2)for bar, val in zip(bars, vals):    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01, f'{val:.3f}', ha='center', va='bottom', fontsize=8)ax = axes[0, 3]vals = [a['peak_mean_val'] for a in agg_list]bars = ax.bar(models_all, vals, color=colours_all, width=0.6)ax.set_title('Peak GPU (MB, lower better)'); ax.set_ylabel('MB')best = int(np.argmin(vals))bars[best].set_edgecolor('black'); bars[best].set_linewidth(2)for bar, val in zip(bars, vals):    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01, f'{val:.0f}', ha='center', va='bottom', fontsize=8)ax = axes[1, 0]x = np.arange(N_LABELS)w = 0.2d_arrays = [dice_3d_all.mean(axis=0)*100,            dice_25_all['axial'].mean(axis=0)*100,            dice_25_all['coronal'].mean(axis=0)*100,            dice_25_all['sagittal'].mean(axis=0)*100]for k, (d_arr, label, color) in enumerate(zip(d_arrays, models_all, colours_all)):    ax.bar(x + (k-1.5)*w, d_arr, w, label=label, color=color, alpha=0.85)ax.set_xticks(x)ax.set_xticklabels([str(l) for l in SEG_LABELS], rotation=45, fontsize=6)ax.set_xlabel('Label ID'); ax.set_ylabel('Dice %')ax.set_title('Per-label Dice'); ax.legend(fontsize=7, ncol=2)ax = axes[1, 1]x = np.arange(2); w = 0.35ax.bar(x-w/2, [n3/1e6, n25/1e6],  w, label='Params (M)',   color='#4C72B0', alpha=0.85)ax.bar(x+w/2, [mb3, mb25],         w, label='Weights (MB)', color='#DD8452', alpha=0.85)ax.set_xticks(x); ax.set_xticklabels(['3D', '2.5D'])ax.set_title('Model size'); ax.legend(fontsize=8)ax = axes[1, 2]pair_x = np.arange(n_pairs)ax.plot(pair_x, [r['dice_mean'] for r in rows_3d], 'o-', color=colours_all[0], label='3D', markersize=5)for orient, color in zip(orient_names, colours_all[1:]):    ax.plot(pair_x, [r['dice_mean'] for r in rows_25[orient]], 's-', color=color,            label=f'2.5D-{orient[:3]}', markersize=4)ax.set_xlabel('Pair index'); ax.set_ylabel('Dice %')ax.set_title('Dice per pair'); ax.legend(fontsize=7)ax.set_xticks(pair_x)ax = axes[1, 3]orient_labels = ['axial', 'coronal', 'sagittal']orient_dice = [agg_25[o]['dice_mean_val'] for o in orient_labels]orient_stds = [agg_25[o]['dice_std_val'] for o in orient_labels]bars = ax.bar(orient_labels, orient_dice, yerr=orient_stds, capsize=5,              color=colours_all[1:], width=0.5)ax.axhline(agg_3d['dice_mean_val'], color=colours_all[0], linestyle='--', linewidth=2, label='3D')ax.set_title('2.5D orientation comparison'); ax.set_ylabel('Dice %')ax.legend(fontsize=8)best = int(np.argmax(orient_dice))bars[best].set_edgecolor('black'); bars[best].set_linewidth(2)for bar, val in zip(bars, orient_dice):    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01, f'{val:.1f}', ha='center', va='bottom', fontsize=9)plt.tight_layout()plt.savefig('model_comparison_metrics.png', dpi=150, bbox_inches='tight')plt.show()

In [ ]:
SHOW_PAIR = 0
i, j = pairings[SHOW_PAIR]
mv = val_moving[i:i+1]
fx = val_fixed[j:j+1]
ms = val_moving_seg[i:i+1]
fs = val_fixed_seg[j:j+1]

best_orient = max(orient_names, key=lambda o: rows_25[o][SHOW_PAIR]['dice_mean'])
best_ax_idx = orient_names.index(best_orient)

with torch.no_grad():
    _, _, wseg_3d_v, fseg_3d_v, _, _ = infer_3d(mv, fx, ms, fs)
    wseg_25_v, fseg_25_v, _, _ = infer_2p5d(mv, fx, ms, fs, axis=best_ax_idx)

if DOWNSAMPLE:
    with torch.no_grad():
        mr_seg_np = preprocess_seg_3d(ms)[0, 0].cpu().numpy().astype(np.int16)
        ct_seg_np = preprocess_seg_3d(fs)[0, 0].cpu().numpy().astype(np.int16)
        mr_np = preprocess_3d(mv)[0, 0].cpu().numpy()
        ct_np = preprocess_3d(fx)[0, 0].cpu().numpy()
        mv_ds = preprocess_3d(mv); fx_ds = preprocess_3d(fx)
        moved_ds, _ = model_3d(mv_ds, fx_ds)
        m3_np = moved_ds[0, 0].cpu().numpy()
    D_vis = mr_np.shape[0]
else:
    mr_seg_np = ms[0, 0].numpy()
    ct_seg_np = fs[0, 0].numpy()
    mr_np = mv[0, 0].numpy()
    ct_np = fx[0, 0].numpy()
    m3_np = None
    D_vis = D

z_mid = D_vis // 2
res_tag = "downsampled" if DOWNSAMPLE else "full res"

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle(f"3D vs 2.5D ({best_orient}): subj {i} -> subj {j}, axial z={z_mid} ({res_tag})",
             fontsize=13, fontweight='bold')

row1_titles = ['Moving seg', 'Fixed seg', '3D warped seg', f'2.5D ({best_orient}) warped seg']
row1_segs = [mr_seg_np[z_mid], ct_seg_np[z_mid], wseg_3d_v[z_mid], wseg_25_v[z_mid]]

for col, (seg, title) in enumerate(zip(row1_segs, row1_titles)):
    ax = axes[0, col]
    im = ax.imshow(seg, cmap='nipy_spectral', interpolation='nearest')
    ax.set_title(title, fontsize=10); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

row2_data = [
    ('MR + moving seg',           mr_np[z_mid], mr_seg_np[z_mid]),
    ('CT + fixed seg',            ct_np[z_mid], ct_seg_np[z_mid]),
    ('3D: moved + warped seg',    m3_np[z_mid], wseg_3d_v[z_mid]),
    (f'2.5D ({best_orient}) + warped seg', m3_np[z_mid], wseg_25_v[z_mid]),
]

for col, (title, img, seg) in enumerate(row2_data):
    ax = axes[1, col]
    ax.imshow(img, cmap='gray')
    seg_mask = (seg > 0).astype(np.float32)
    if seg_mask.sum() > 0:
        ax.contour(seg_mask, levels=[0.5], colors='lime', linewidths=0.5)
    for lbl, color in zip([2, 4, 10, 16], ['red', 'cyan', 'yellow', 'magenta']):
        lbl_mask = (seg == lbl).astype(np.float32)
        if lbl_mask.sum() > 0:
            ax.contour(lbl_mask, levels=[0.5], colors=color, linewidths=0.8)
    ax.set_title(title, fontsize=9); ax.axis('off')

plt.tight_layout()
plt.savefig('model_comparison_segs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best_orient_vis = max(orient_names, key=lambda o: rows_25[o][SHOW_PAIR]['dice_mean'])best_ax_idx = orient_names.index(best_orient_vis)ax_idx_val, D_ax_val, _, _ = ORIENT_CONFIG[best_ax_idx]m25_np = np.copy(mr_np)with torch.no_grad():    slice_indices = list(range(WR, D_ax_val - WR))    for start in range(0, len(slice_indices), 16):        end = min(start + 16, len(slice_indices))        mr_b = np.stack([np.ascontiguousarray(_extract_stack(mr_np, ax_idx_val, slice_indices[k], WR))                         for k in range(start, end)])        ct_b = np.stack([np.ascontiguousarray(_extract_stack(ct_np, ax_idx_val, slice_indices[k], WR))                         for k in range(start, end)])        mr_t = torch.from_numpy(mr_b).to(device)        ct_t = torch.from_numpy(ct_b).to(device)        moved_slices, _ = model_2p5d(mr_t, ct_t)        moved_np_batch = moved_slices[:, 0].cpu().numpy()        for bi, k in enumerate(range(start, end)):            z_idx = slice_indices[k]            if ax_idx_val == 0:                m25_np[z_idx] = moved_np_batch[bi]            elif ax_idx_val == 1:                m25_np[:, z_idx, :] = moved_np_batch[bi]            else:                m25_np[:, :, z_idx] = moved_np_batch[bi]z = D_vis // 2err_3d = np.abs(ct_np[z] - m3_np[z])err_25 = np.abs(ct_np[z] - m25_np[z])err_unr = np.abs(ct_np[z] - mr_np[z])vmax = max(err_3d.max(), err_25.max(), err_unr.max())fig, axes = plt.subplots(1, 4, figsize=(16, 4))res_tag = "downsampled" if DOWNSAMPLE else "full res"fig.suptitle(f"Absolute error vs Fixed CT (axial z={z}, {res_tag})",             fontsize=12, fontweight='bold')for ax, img, title in zip(axes,        [err_unr, err_3d, err_25, ct_np[z]],        [f'Unregistered  MSE={np.mean(err_unr**2):.5f}',         f'3D moved      MSE={np.mean(err_3d**2):.5f}',         f'2.5D ({best_orient_vis})  MSE={np.mean(err_25**2):.5f}',         'Fixed CT (reference)']):    cmap = 'hot' if 'Unreg' in title or 'moved' in title or '2.5D' in title else 'gray'    vmax_use = vmax if cmap == 'hot' else None    im = ax.imshow(img, cmap=cmap, vmin=0, vmax=vmax_use)    ax.set_title(title, fontsize=9); ax.axis('off')    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)plt.tight_layout()plt.savefig('model_comparison_error.png', dpi=150, bbox_inches='tight')plt.show()

In [ ]:
print("3D")
print(f"  Params: {n3/1e6:.3f}M  |  Weights: {mb3:.2f}MB")
print(f"  Dice: {agg_3d['Dice_%']}  |  HD95: {agg_3d['HD95']}")
print(f"  |J|<=0: {agg_3d['|J|<=0']}  |  Time: {agg_3d['Time_s']}s  |  GPU: {agg_3d['PeakGPU_MB']}MB")

for orient in orient_names:
    a = agg_25[orient]
    print(f"\n2.5D ({orient})")
    print(f"  Params: {n25/1e6:.3f}M  |  Dice: {a['Dice_%']}  |  HD95: {a['HD95']}")
    print(f"  Time: {a['Time_s']}s  |  GPU: {a['PeakGPU_MB']}MB")

best_orient = max(orient_names, key=lambda o: agg_25[o]['dice_mean_val'])
print(f"\nBest 2.5D orientation: {best_orient} (Dice {agg_25[best_orient]['dice_mean_val']:.2f}%)")
print(f"{n_pairs} cross-subject pairs, {N_LABELS} labels, seed=42")